# 🥬 LettuceDetect × HaluEval: Hallucination Detection Training & Evaluation

This notebook trains LettuceDetect (a token-level hallucination detector) on the **HaluEval** dataset and produces comprehensive performance analysis.

**Make sure to select GPU runtime**: Runtime → Change runtime type → T4 GPU

## 1️⃣ Setup & Installation

In [ ]:
# Clone LettuceDetect and install
!git clone https://github.com/KRLabsOrg/LettuceDetect.git
%cd LettuceDetect
!pip install -e . -q
!pip install matplotlib seaborn -q

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## 2️⃣ Download HaluEval Dataset

In [ ]:
import os
os.makedirs("data/halueval", exist_ok=True)

!wget -q -O data/halueval/qa_data.json https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/qa_data.json
!wget -q -O data/halueval/dialogue_data.json https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/dialogue_data.json
!wget -q -O data/halueval/summarization_data.json https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/summarization_data.json

# Verify downloads
for f in ["qa_data.json", "dialogue_data.json", "summarization_data.json"]:
    size = os.path.getsize(f"data/halueval/{f}") / 1e6
    print(f"  ✅ {f}: {size:.1f} MB")

## 3️⃣ Patch LettuceDetect to Accept HaluEval Dataset

In [ ]:
# Patch HallucinationSample to accept 'halueval' as a dataset type
import lettucedetect.datasets.hallucination_dataset as hd
import inspect

# Monkey-patch: remove the strict Literal type check by redefining the dataclass
from dataclasses import dataclass
from typing import Literal

@dataclass
class HallucinationSample:
    prompt: str
    answer: str
    labels: list
    split: str  # 'train', 'dev', 'test'
    task_type: str
    dataset: str  # 'ragtruth', 'ragbench', 'halueval'
    language: str  # 'en', 'de'

    def to_json(self):
        return {
            "prompt": self.prompt,
            "answer": self.answer,
            "labels": self.labels,
            "split": self.split,
            "task_type": self.task_type,
            "dataset": self.dataset,
            "language": self.language,
        }

    @classmethod
    def from_json(cls, json_dict):
        return cls(
            prompt=json_dict["prompt"],
            answer=json_dict["answer"],
            labels=json_dict["labels"],
            split=json_dict["split"],
            task_type=json_dict["task_type"],
            dataset=json_dict["dataset"],
            language=json_dict["language"],
        )

# Apply patch
hd.HallucinationSample = HallucinationSample
print("✅ Patched HallucinationSample to accept 'halueval' dataset")

## 4️⃣ Preprocess HaluEval → LettuceDetect Format

In [ ]:
import json
import random
from pathlib import Path
from lettucedetect.datasets.hallucination_dataset import HallucinationData

random.seed(42)

def load_jsonl(filepath):
    """Load JSONL file (one JSON object per line)."""
    with open(filepath, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def process_qa(data, split):
    samples = []
    for item in data:
        prompt = f"{item.get('knowledge', '')}\n\nQuestion: {item.get('question', '')}"
        right = item.get('right_answer', '')
        if right:
            samples.append(HallucinationSample(prompt, right, [], split, 'qa', 'halueval', 'en'))
        halluc = item.get('hallucinated_answer', '')
        if halluc:
            samples.append(HallucinationSample(prompt, halluc, [{'start': 0, 'end': len(halluc), 'label': 'hallucination'}], split, 'qa', 'halueval', 'en'))
    return samples

def process_dialogue(data, split):
    samples = []
    for item in data:
        prompt = f"{item.get('knowledge', '')}\n\nDialogue History: {item.get('dialogue_history', '')}"
        right = item.get('right_response', '')
        if right:
            samples.append(HallucinationSample(prompt, right, [], split, 'dialogue', 'halueval', 'en'))
        halluc = item.get('hallucinated_response', '')
        if halluc:
            samples.append(HallucinationSample(prompt, halluc, [{'start': 0, 'end': len(halluc), 'label': 'hallucination'}], split, 'dialogue', 'halueval', 'en'))
    return samples

def process_summarization(data, split):
    samples = []
    for item in data:
        prompt = item.get('document', '')
        right = item.get('right_summary', '')
        if right:
            samples.append(HallucinationSample(prompt, right, [], split, 'summarization', 'halueval', 'en'))
        halluc = item.get('hallucinated_summary', '')
        if halluc:
            samples.append(HallucinationSample(prompt, halluc, [{'start': 0, 'end': len(halluc), 'label': 'hallucination'}], split, 'summarization', 'halueval', 'en'))
    return samples

# Process all task types
MAX_SAMPLES = 2000  # Per task type (use all 10K per task for full training, or cap for speed)
TRAIN_RATIO = 0.8

all_samples = []
tasks = {
    'qa': ('data/halueval/qa_data.json', process_qa),
    'dialogue': ('data/halueval/dialogue_data.json', process_dialogue),
    'summarization': ('data/halueval/summarization_data.json', process_summarization),
}

for name, (path, processor) in tasks.items():
    data = load_jsonl(path)
    if len(data) > MAX_SAMPLES:
        random.shuffle(data)
        data = data[:MAX_SAMPLES]
    random.shuffle(data)
    split_idx = int(len(data) * TRAIN_RATIO)
    train_samples = processor(data[:split_idx], 'train')
    test_samples = processor(data[split_idx:], 'test')
    all_samples.extend(train_samples + test_samples)
    print(f"  {name}: {len(train_samples)} train + {len(test_samples)} test")

hall_data = HallucinationData(samples=all_samples)
train_ct = sum(1 for s in all_samples if s.split == 'train')
test_ct = sum(1 for s in all_samples if s.split == 'test')
hall_ct = sum(1 for s in all_samples if len(s.labels) > 0)

with open('data/halueval/halueval_data.json', 'w') as f:
    json.dump(hall_data.to_json(), f, indent=2)

print(f"\n📊 Total: {len(all_samples)} samples")
print(f"   Train: {train_ct} | Test: {test_ct}")
print(f"   Hallucinated: {hall_ct} | Supported: {len(all_samples) - hall_ct}")
print("✅ Saved to data/halueval/halueval_data.json")

## 5️⃣ Train the Model 🚀

In [ ]:
import json
import random
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
)
from lettucedetect.datasets.hallucination_dataset import HallucinationData, HallucinationDataset
from lettucedetect.models.trainer import Trainer

# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)

# Load preprocessed data
data_path = Path('data/halueval/halueval_data.json')
hall_data = HallucinationData.from_json(json.loads(data_path.read_text()))

# Split train/dev
train_samples = [s for s in hall_data.samples if s.split == 'train']
random.shuffle(train_samples)
dev_size = int(len(train_samples) * 0.1)
dev_samples = train_samples[-dev_size:]
train_samples = train_samples[:-dev_size]

print(f"Train: {len(train_samples)}, Dev: {len(dev_samples)}")

# Setup tokenizer and model
MODEL_NAME = 'answerdotai/ModernBERT-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)

train_dataset = HallucinationDataset(train_samples, tokenizer)
dev_dataset = HallucinationDataset(dev_samples, tokenizer)

# ⚡ Training hyperparameters
BATCH_SIZE = 8
EPOCHS = 6
LEARNING_RATE = 1e-5
OUTPUT_DIR = 'output/halueval_detector'

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=2, trust_remote_code=True)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    test_loader=dev_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    save_path=OUTPUT_DIR,
)

print(f"\n🚀 Starting training on {next(model.parameters()).device}")
print(f"   Steps per epoch: {len(train_loader)}")
print(f"   Total steps: {len(train_loader) * EPOCHS}")

trainer.train()
print("\n✅ Training complete! Model saved to", OUTPUT_DIR)

## 6️⃣ Comprehensive Evaluation & Performance Analysis 📊

In [ ]:
import torch
import json
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
)
from sklearn.metrics import (
    accuracy_score, classification_report,
    precision_recall_fscore_support, roc_curve, auc,
    confusion_matrix, precision_recall_curve,
)
from lettucedetect.datasets.hallucination_dataset import HallucinationData, HallucinationDataset

# Load model and test data
MODEL_PATH = 'output/halueval_detector'
DATA_PATH = 'data/halueval/halueval_data.json'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH, trust_remote_code=True).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)

hall_data = HallucinationData.from_json(json.loads(Path(DATA_PATH).read_text()))
test_samples = [s for s in hall_data.samples if s.split == 'test']

print(f"📦 Test samples: {len(test_samples)}")
print(f"   Hallucinated: {sum(1 for s in test_samples if len(s.labels) > 0)}")
print(f"   Supported: {sum(1 for s in test_samples if len(s.labels) == 0)}")

test_dataset = HallucinationDataset(test_samples, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=data_collator)

In [ ]:
# ── Token-Level Evaluation ──
model.eval()
all_preds, all_labels, all_probs = [], [], []

# Also collect example-level predictions
example_preds, example_labels, example_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
        )
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(logits, dim=-1)

        # Token-level
        mask = batch['labels'] != -100
        all_preds.extend(predictions[mask].cpu().numpy().tolist())
        all_labels.extend(batch['labels'][mask].cpu().numpy().tolist())
        all_probs.extend(probs[:, :, 1][mask].cpu().numpy().tolist())

        # Example-level
        for i in range(batch['labels'].size(0)):
            sample_labels = batch['labels'][i]
            sample_preds = predictions[i].cpu()
            valid_mask = sample_labels != -100
            if valid_mask.sum() == 0:
                example_labels.append(0); example_preds.append(0); example_probs.append(0.0)
            else:
                sl = sample_labels[valid_mask].cpu()
                sp = sample_preds[valid_mask]
                sprobs = probs[i][valid_mask]
                example_labels.append(1 if (sl == 1).any().item() else 0)
                example_preds.append(1 if (sp == 1).any().item() else 0)
                example_probs.append(sprobs[:, 1].max().item())

print("Evaluation complete!")

In [ ]:
# ── Print Token-Level Results ──
precision_t, recall_t, f1_t, _ = precision_recall_fscore_support(all_labels, all_preds, labels=[0, 1], average=None, zero_division=0)
acc_t = accuracy_score(all_labels, all_preds)
fpr_t, tpr_t, _ = roc_curve(all_labels, all_probs)
auroc_t = auc(fpr_t, tpr_t)
cm_t = confusion_matrix(all_labels, all_preds, labels=[0, 1])

print("="*60)
print("📊 TOKEN-LEVEL EVALUATION")
print("="*60)
report_t = classification_report(all_labels, all_preds, target_names=['Supported', 'Hallucinated'], digits=4, zero_division=0)
print(report_t)
print(f"Accuracy:  {acc_t:.4f}")
print(f"AUROC:     {auroc_t:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN={cm_t[0][0]:>5}  FP={cm_t[0][1]:>5}")
print(f"  FN={cm_t[1][0]:>5}  TP={cm_t[1][1]:>5}")

In [ ]:
# ── Print Example-Level Results ──
precision_e, recall_e, f1_e, _ = precision_recall_fscore_support(example_labels, example_preds, labels=[0, 1], average=None, zero_division=0)
acc_e = accuracy_score(example_labels, example_preds)
fpr_e, tpr_e, _ = roc_curve(example_labels, example_probs)
auroc_e = auc(fpr_e, tpr_e)
cm_e = confusion_matrix(example_labels, example_preds, labels=[0, 1])

print("="*60)
print("📊 EXAMPLE-LEVEL EVALUATION")
print("="*60)
report_e = classification_report(example_labels, example_preds, target_names=['Supported', 'Hallucinated'], digits=4, zero_division=0)
print(report_e)
print(f"Accuracy:  {acc_e:.4f}")
print(f"AUROC:     {auroc_e:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN={cm_e[0][0]:>5}  FP={cm_e[0][1]:>5}")
print(f"  FN={cm_e[1][0]:>5}  TP={cm_e[1][1]:>5}")

In [ ]:
# ── Per-Task Breakdown ──
task_map = {}
for s in test_samples:
    task_map.setdefault(s.task_type, []).append(s)

for task_type, samples in task_map.items():
    task_dataset = HallucinationDataset(samples, tokenizer)
    task_loader = DataLoader(task_dataset, batch_size=8, shuffle=False, collate_fn=data_collator)

    task_ex_preds, task_ex_labels, task_ex_probs = [], [], []
    with torch.no_grad():
        for batch in task_loader:
            outputs = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
            logits = outputs.logits
            probs_batch = torch.softmax(logits, dim=-1)
            preds_batch = torch.argmax(logits, dim=-1)
            for i in range(batch['labels'].size(0)):
                sl = batch['labels'][i]
                sp = preds_batch[i].cpu()
                vm = sl != -100
                if vm.sum() == 0:
                    task_ex_labels.append(0); task_ex_preds.append(0); task_ex_probs.append(0.0)
                else:
                    sl_v = sl[vm].cpu(); sp_v = sp[vm]; sp_probs = probs_batch[i][vm]
                    task_ex_labels.append(1 if (sl_v == 1).any().item() else 0)
                    task_ex_preds.append(1 if (sp_v == 1).any().item() else 0)
                    task_ex_probs.append(sp_probs[:, 1].max().item())

    p, r, f, _ = precision_recall_fscore_support(task_ex_labels, task_ex_preds, labels=[0, 1], average=None, zero_division=0)
    acc = accuracy_score(task_ex_labels, task_ex_preds)
    print(f"\n{'='*50}")
    print(f"📋 Task: {task_type.upper()} ({len(samples)} samples)")
    print(f"{'='*50}")
    print(classification_report(task_ex_labels, task_ex_preds, target_names=['Supported', 'Hallucinated'], digits=4, zero_division=0))
    print(f"Accuracy: {acc:.4f}")

## 7️⃣ Performance Visualization 📈

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ─── ROC Curve: Token Level ───
ax = axes[0][0]
ax.plot(fpr_t, tpr_t, color='#e74c3c', lw=2.5, label=f'Token ROC (AUC = {auroc_t:.4f})')
ax.plot([0, 1], [0, 1], color='#bdc3c7', lw=1.5, linestyle='--', alpha=0.7)
ax.fill_between(fpr_t, tpr_t, alpha=0.1, color='#e74c3c')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Token-Level ROC Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

# ─── ROC Curve: Example Level ───
ax = axes[0][1]
ax.plot(fpr_e, tpr_e, color='#3498db', lw=2.5, label=f'Example ROC (AUC = {auroc_e:.4f})')
ax.plot([0, 1], [0, 1], color='#bdc3c7', lw=1.5, linestyle='--', alpha=0.7)
ax.fill_between(fpr_e, tpr_e, alpha=0.1, color='#3498db')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Example-Level ROC Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

# ─── Metrics Bar Chart ───
ax = axes[1][0]
metrics = ['Precision', 'Recall', 'F1']
token_vals = [precision_t[1], recall_t[1], f1_t[1]]
example_vals = [precision_e[1], recall_e[1], f1_e[1]]
x = np.arange(len(metrics))
width = 0.3
bars1 = ax.bar(x - width/2, token_vals, width, label='Token-Level', color='#e74c3c', alpha=0.85)
bars2 = ax.bar(x + width/2, example_vals, width, label='Example-Level', color='#3498db', alpha=0.85)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Hallucination Detection Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
for b in bars1 + bars2:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# ─── Confusion Matrix Heatmap ───
ax = axes[1][1]
import seaborn as sns
sns.heatmap(cm_e, annot=True, fmt='d', cmap='Blues', xticklabels=['Supported', 'Hallucinated'],
            yticklabels=['Supported', 'Hallucinated'], ax=ax, annot_kws={'size': 14})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Example-Level Confusion Matrix', fontsize=14, fontweight='bold')

plt.suptitle('LettuceDetect on HaluEval - Performance Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/performance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Performance plots saved to output/performance_analysis.png")

In [ ]:
# ─── Final Summary Table ───
print("\n" + "="*65)
print("   🥬 LETTUCEDETECT × HALUEVAL — PERFORMANCE SUMMARY")
print("="*65)
print(f"\n{'Metric':<28} {'Token-Level':>12} {'Example-Level':>14}")
print("-"*58)
print(f"{'Hallucinated F1':<28} {f1_t[1]:>12.4f} {f1_e[1]:>14.4f}")
print(f"{'Hallucinated Precision':<28} {precision_t[1]:>12.4f} {precision_e[1]:>14.4f}")
print(f"{'Hallucinated Recall':<28} {recall_t[1]:>12.4f} {recall_e[1]:>14.4f}")
print(f"{'Supported F1':<28} {f1_t[0]:>12.4f} {f1_e[0]:>14.4f}")
print(f"{'Supported Precision':<28} {precision_t[0]:>12.4f} {precision_e[0]:>14.4f}")
print(f"{'Supported Recall':<28} {recall_t[0]:>12.4f} {recall_e[0]:>14.4f}")
print(f"{'Accuracy':<28} {acc_t:>12.4f} {acc_e:>14.4f}")
print(f"{'AUROC':<28} {auroc_t:>12.4f} {auroc_e:>14.4f}")
print("="*58)

# Save results to JSON
results = {
    'token_level': {
        'supported': {'precision': float(precision_t[0]), 'recall': float(recall_t[0]), 'f1': float(f1_t[0])},
        'hallucinated': {'precision': float(precision_t[1]), 'recall': float(recall_t[1]), 'f1': float(f1_t[1])},
        'accuracy': float(acc_t), 'auroc': float(auroc_t),
    },
    'example_level': {
        'supported': {'precision': float(precision_e[0]), 'recall': float(recall_e[0]), 'f1': float(f1_e[0])},
        'hallucinated': {'precision': float(precision_e[1]), 'recall': float(recall_e[1]), 'f1': float(f1_e[1])},
        'accuracy': float(acc_e), 'auroc': float(auroc_e),
    },
}
with open('output/evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\n✅ Results saved to output/evaluation_results.json")

## 8️⃣ Download Results

In [ ]:
# Download the trained model and results
!zip -r halueval_results.zip output/halueval_detector output/evaluation_results.json output/performance_analysis.png

from google.colab import files
files.download('halueval_results.zip')
print("\n📥 Download complete! The zip contains:")
print("  - output/halueval_detector/ (trained model)")
print("  - output/evaluation_results.json (all metrics)")
print("  - output/performance_analysis.png (ROC curves & charts)")